In [49]:
# ==========================================
# 1. Imports and Configuration
# ==========================================

# Core data manipulation and numerical operations
import pandas as pd
import numpy as np

# Modeling utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest

# Scientific computing
from scipy.stats import uniform

# Bayesian Linear Regression model
# ARDRegression is excellent for handling datasets where many features might be irrelevant
from sklearn.linear_model import ARDRegression

# Bayesian Hyperparameter Optimization
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# Project Configuration
import warnings
warnings.filterwarnings("ignore")

# Set seed for reproducibility if needed later
RANDOM_STATE = 42

In [50]:
# ==========================================
# 2. Data Loading
# ==========================================

import os

# Define the data path.
# the data file is in the 'data/' folder relative to this notebook.
DATA_PATH = '/kaggle/input/rmb-cpi-aug/downloadcpiAug.xlsx' #'data/downloadcpiAug.xlsx' 

def load_cpi_data(file_path: str) -> pd.DataFrame:
    """
    Loads the CPI dataset from the specified Excel file path.

    Args:
        file_path (str): The local or absolute path to the .xlsx file.

    Returns:
        pd.DataFrame: The loaded CPI dataset.
        
    Raises:
        FileNotFoundError: If the dataset is not found at the specified path.
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}. Please check your path.")
    
    try:
        df = pd.read_excel(file_path)
        print(f"Data loaded successfully. Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return pd.DataFrame()



In [51]:
# Execute data loading
download_cpi = load_cpi_data(DATA_PATH)

# Display the first few rows to confirm structure
download_cpi.head()

Data loaded successfully. Shape: (746, 198)


,H01,H02,H03,H04,H05,H06,H13,H17,H18,H24,...,MO102022,MO112022,MO122022,MO012023,MO022023,MO032023,MO042023,MO052023,MO062023,MO072023
0,P0141,Consumer Price Index,CPA00000,All Items,NaN,NaN,Western Cape,Index,Dec 2021 = 100,2008 01,...,106.3,106.6,106.9,106.9,107.7,108.8,109.4,109.6,109.8,110.7
1,P0141,Consumer Price Index,CPA01000,Food and non alcoholic beverages,NaN,NaN,Western Cape,Index,Dec 2021 = 100,2008 01,...,110.3,110.9,111.3,113.5,114.0,115.4,116.0,116.5,116.9,117.1
2,P0141,Consumer Price Index,CPA01100,Food,NaN,NaN,Western Cape,Index,Dec 2021 = 100,2008 01,...,110.2,110.9,111.2,113.6,113.9,115.3,115.9,116.4,116.9,117.2
3,P0141,Consumer Price Index,CPA01110,Bread and cereals,NaN,NaN,Western Cape,Index,Dec 2021 = 100,2008 01,...,117.1,117.2,118.2,119.4,120.6,121.6,122.5,124.5,124.5,125.1
4,P0141,Consumer Price Index,CPA01120,Meat,NaN,NaN,Western Cape,Index,Dec 2021 = 100,2008 01,...,108.1,109.3,109.7,112.6,112.1,112.0,111.9,111.5,112.0,110.8


In [52]:
# ==========================================
# 3. Data Filtering and Cleaning
# ==========================================

# List of target categories required for the nowcasting competition
TARGET_CATEGORIES = [
    'CPI Headline',
    'Food and non alcoholic beverages',
    'Alcoholic beverages and tobacco',
    'Clothing and footwear',
    'Housing and utilities',
    'Household contents and equipment',
    'Health',
    'Transport',
    'Communication',
    'Recreation and culture',
    'Education',
    'Restaurants and hotels',
    'Miscellaneous goods and services'
]

def filter_cpi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters the raw CPI dataframe for the specific categories and urban areas.
    Renames columns for better interpretability.

    Args:
        df (pd.DataFrame): The raw dataframe from Stats SA.

    Returns:
        pd.DataFrame: A filtered and renamed dataframe.
    """
    # Filtering based on specific Stats SA codes:
    # H04: Category name
    # H13: Geographical area (All urban areas is the national benchmark)
    df_filtered = df[
        (df['H04'].isin(TARGET_CATEGORIES)) & 
        (df['H13'] == 'All urban areas')
    ].copy()

    # Mapping internal codes to human-readable names for cleaner code downstream
    rename_map = {
        'H04': 'Category',
    }
    
    # Selecting and renaming specific columns (Adjust these based on your specific Excel structure)
    df_filtered = df_filtered.rename(columns=rename_map)

    return df_filtered

# Apply the filter
cpi_filtered = filter_cpi_data(download_cpi)

# Validate results
print(f"Categories selected: {len(cpi_filtered['Category'].unique())} of 13")
print(cpi_filtered['Category'].unique())

Categories selected: 13 of 13
['CPI Headline' 'Education' 'Food and non alcoholic beverages'
 'Alcoholic beverages and tobacco' 'Clothing and footwear'
 'Housing and utilities' 'Household contents and equipment' 'Health'
 'Transport' 'Communication' 'Recreation and culture'
 'Restaurants and hotels' 'Miscellaneous goods and services']


In [53]:
# ==========================================
# 4. Refining Category Granularity
# ==========================================

def refine_education_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters out specific sub-categories within the 'Education' group to 
    ensure we are only using the primary 'Education' index.
    
    This prevents double-counting or using inconsistent sub-indices.

    Args:
        df (pd.DataFrame): The filtered CPI dataframe.

    Returns:
        pd.DataFrame: Dataframe with unwanted education sub-categories removed.
    """
    # List of sub-categories to remove as identified in the H05 column
    # These represent granular components that might deviate from the main index
    unwanted_edu_subcats = [
        'University boarding fees',
        'Tertiary education',
        'Primary and secondary education',
        'Education including boarding fees'
    ]
    
    df_refined = df[~(df['H05'].isin(unwanted_edu_subcats))].copy()
    
    initial_count = len(df)
    final_count = len(df_refined)
    
    print(f"Refinement complete: Removed {initial_count - final_count} sub-category rows.")
    
    return df_refined


edu_filtered = refine_education_category(cpi_filtered)

Refinement complete: Removed 4 sub-category rows.


In [54]:
edu_filtered

,H01,H02,H03,Category,H05,H06,H13,H17,H18,H24,...,MO102022,MO112022,MO122022,MO012023,MO022023,MO032023,MO042023,MO052023,MO062023,MO072023
557,P0141,Consumer Price Index,CPS00000,CPI Headline,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,106.5,106.8,107.2,107.1,107.9,109.0,109.4,109.6,109.8,110.8
578,P0141,Consumer Price Index,CPS01000,Food and non alcoholic beverages,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,111.3,111.9,112.4,114.4,115.6,116.7,117.4,117.7,118.3,118.5
592,P0141,Consumer Price Index,CPS02000,Alcoholic beverages and tobacco,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,106.1,106.4,106.2,106.5,106.9,109.2,110.2,110.6,110.9,111.5
598,P0141,Consumer Price Index,CPS03000,Clothing and footwear,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,102.6,102.6,102.8,102.9,103.5,103.4,103.7,104.1,104.3,104.5
601,P0141,Consumer Price Index,CPS04000,Housing and utilities,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,103.8,103.8,104.1,104.1,104.1,104.5,104.6,104.6,105.4,108.4
613,P0141,Consumer Price Index,CPS05000,Household contents and equipment,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,105.0,105.2,106.1,106.6,106.6,107.8,107.7,107.5,107.7,108.2
618,P0141,Consumer Price Index,CPS06000,Health,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,104.5,104.7,104.8,104.9,108.5,109.1,109.6,110.3,110.8,110.6
625,P0141,Consumer Price Index,CPS07000,Transport,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,111.7,113.1,113.9,109.9,110.6,112.9,113.1,113.3,112.3,112.6
634,P0141,Consumer Price Index,CPS08000,Communication,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,99.8,99.2,99.1,99.4,99.8,99.7,99.8,99.7,99.6,99.5
637,P0141,Consumer Price Index,CPS09000,Recreation and culture,NaN,NaN,All urban areas,Index,Dec 2021 = 100,2008 01,...,102.7,102.8,103.4,103.4,103.3,104.2,104.9,104.8,105.3,105.5


In [55]:
#Dropping extra features that aren't needed
edu_filtered = edu_filtered.drop(['H01','H02','H03','H05','H06','H13',
                                  'H17','H18','H24','H25'],axis=1)


In [56]:
# ==========================================
# 5. Time-Series Structural Alignment
# ==========================================

def format_cpi_columns(df: pd.DataFrame, start_date: str = "2008-01-01", num_months: int = 187) -> pd.DataFrame:
    """
    Generates a standardized date range and assigns it as column headers.
    
    Args:
        df (pd.DataFrame): The filtered CPI dataframe.
        start_date (str): The starting month of the dataset.
        num_months (int): Total number of months up to the forecasting horizon 
                          (July 2023 = 187 months from Jan 2008).

    Returns:
        pd.DataFrame: Dataframe with formatted date columns.
    """
    # Generate a sequence of dates (end of month frequency)
    date_index = pd.date_range(start=start_date, periods=num_months, freq='M')

    
    formatted_dates = [f"{d.month}-{d.year}" for d in date_index]

    # Create the new column list
    new_columns = ['Category'] + formatted_dates
    
    # Validation: Ensure column count matches data shape
    if len(new_columns) != df.shape[1]:
        print(f"Warning: Column mismatch. Data has {df.shape[1]} columns, but we generated {len(new_columns)}.")
    
    df.columns = new_columns
    return df

In [57]:
# Execute column renaming
# We expect 187 months (Jan 2008 to July 2023) + 1 'Category' column = 188 columns
edu_filtered = format_cpi_columns(edu_filtered)

# Check the last few columns to ensure it ends at 7-2023
print(f"Dataset structure: {edu_filtered.shape}")
print(f"Last column: {edu_filtered.columns[-1]}")

Dataset structure: (13, 188)
Last column: 7-2023


In [58]:
edu_filtered

,Category,1-2008,2-2008,3-2008,4-2008,5-2008,6-2008,7-2008,8-2008,9-2008,...,10-2022,11-2022,12-2022,1-2023,2-2023,3-2023,4-2023,5-2023,6-2023,7-2023
557,CPI Headline,48.6,49.0,49.7,49.9,50.3,51.0,51.7,52.0,52.3,...,106.5,106.8,107.2,107.1,107.9,109.0,109.4,109.6,109.8,110.8
578,Food and non alcoholic beverages,42.9,42.9,43.5,44.1,44.9,45.6,46.3,46.9,47.6,...,111.3,111.9,112.4,114.4,115.6,116.7,117.4,117.7,118.3,118.5
592,Alcoholic beverages and tobacco,41.1,41.6,43.4,43.7,43.8,43.8,43.9,44.6,45.0,...,106.1,106.4,106.2,106.5,106.9,109.2,110.2,110.6,110.9,111.5
598,Clothing and footwear,65.7,65.9,66.1,66.1,66.3,66.6,66.7,67.1,67.3,...,102.6,102.6,102.8,102.9,103.5,103.4,103.7,104.1,104.3,104.5
601,Housing and utilities,46.1,46.1,46.8,46.9,47.0,47.8,48.9,49.3,49.7,...,103.8,103.8,104.1,104.1,104.1,104.5,104.6,104.6,105.4,108.4
613,Household contents and equipment,68.3,68.1,68.5,68.9,69.0,69.7,69.8,70.2,70.8,...,105.0,105.2,106.1,106.6,106.6,107.8,107.7,107.5,107.7,108.2
618,Health,45.2,47.4,47.4,47.4,47.8,47.9,48.1,48.2,48.2,...,104.5,104.7,104.8,104.9,108.5,109.1,109.6,110.3,110.8,110.6
625,Transport,52.5,52.6,53.6,54.3,54.9,56.7,57.9,57.3,56.9,...,111.7,113.1,113.9,109.9,110.6,112.9,113.1,113.3,112.3,112.6
634,Communication,105.4,105.3,105.2,104.8,104.9,104.9,104.8,106.1,105.9,...,99.8,99.2,99.1,99.4,99.8,99.7,99.8,99.7,99.6,99.5
637,Recreation and culture,68.6,68.6,68.6,69.3,70.4,70.4,70.6,72.0,72.2,...,102.7,102.8,103.4,103.4,103.3,104.2,104.9,104.8,105.3,105.5


In [59]:
# ==========================================
# 6. Reshaping to Long Format
# ==========================================

def reshape_cpi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivots the dataframe from wide (months as columns) to long (months as rows).
    Converts dates to datetime objects and sorts the data chronologically.

    Args:
        df (pd.DataFrame): The wide-format CPI dataframe.

    Returns:
        pd.DataFrame: A cleaned, long-format dataframe sorted by Category and Date.
    """
    # 1. Melt the data: 'Category' stays, all date columns become rows in 'Date'
    df_long = pd.melt(df, id_vars='Category', var_name='Date', value_name='Value')

    # 2. Convert 'Date' column to datetime objects
    # Our format was 'Month-Year' (e.g., '1-2008')
    df_long['Date'] = pd.to_datetime(df_long['Date'], format='%m-%Y')

    # 3. Ensure 'Value' is numeric (important for modeling)
    df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

    # 4. Sort the data
    # Sorting by Category then Date ensures a clean sequence for feature engineering
    df_long = df_long.sort_values(by=['Category', 'Date']).reset_index(drop=True)

    return df_long

# Apply the transformation
cpi_long = reshape_cpi_data(edu_filtered)

# Preview the results
print(f"Long format shape: {cpi_long.shape}")
print(cpi_long.head())

Long format shape: (2431, 3)
                          Category       Date  Value
0  Alcoholic beverages and tobacco 2008-01-01   41.1
1  Alcoholic beverages and tobacco 2008-02-01   41.6
2  Alcoholic beverages and tobacco 2008-03-01   43.4
3  Alcoholic beverages and tobacco 2008-04-01   43.7
4  Alcoholic beverages and tobacco 2008-05-01   43.8


In [60]:
# ==========================================
# 7. Feature Engineering (Lagged Observations)
# ==========================================

def create_lagged_features(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """
    Creates auto-regressive features by shifting the CPI values.
    
    This function treats each CPI category as an independent time series 
    to prevent data leakage between different categories.

    Args:
        df (pd.DataFrame): Long-format dataframe (Expected order: Descending by Date).
        steps (int): The number of previous months to use as features.

    Returns:
        pd.DataFrame: Dataframe with new columns 'Value_1', 'Value_2', etc., 
                      representing past observations.
    """
    data = df.copy()
    
    # We group by 'Category' to ensure shifts happen within the same CPI class.
    # We use .shift(-i) because the data is sorted with the newest dates at the top.
    for i in range(1, steps + 1):
        data[f'Value_{i}'] = data.groupby('Category')['Value'].shift(-i)

    # Drop rows with NaN values created by the shift (the oldest 'steps' months)
    data = data.dropna(axis=0)
    
    # Reset index for a clean slate before modeling
    data = data.reset_index(drop=True)
    
    print(f"Feature engineering complete. Created {steps} lag features.")
    print(f"New shape: {data.shape}")
    
    return data

# Generating 12 months of lags (1 year of historical context for each prediction)
LAG_STEPS = 15
cpi_with_lags = create_lagged_features(cpi_values, steps=LAG_STEPS)

# Preview the head to see Value_1, Value_2... alongside the target 'Value'
cpi_with_lags.head()

Feature engineering complete. Created 15 lag features.
New shape: (2236, 18)


,Category,Date,Value,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,Value_8,Value_9,Value_10,Value_11,Value_12,Value_13,Value_14,Value_15
0,Miscellaneous goods and services,7-2023,109.9,109.6,109.4,109.3,107.9,107.7,105.4,104.9,104.7,104.7,103.8,103.5,103.4,103.0,102.9,102.8
1,Restaurants and hotels,7-2023,110.0,110.0,110.4,108.6,109.6,108.8,106.8,107.8,108.0,107.4,106.2,104.3,104.6,104.2,103.8,103.3
2,Education,7-2023,110.4,110.4,110.4,110.4,110.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4
3,Recreation and culture,7-2023,105.5,105.3,104.8,104.9,104.2,103.3,103.4,103.4,102.8,102.7,102.5,102.4,102.3,101.2,101.0,100.7
4,Communication,7-2023,99.5,99.6,99.7,99.8,99.7,99.8,99.4,99.1,99.2,99.8,99.9,100.1,99.9,100.3,99.9,99.6


In [61]:
# ==========================================
# 8. Hyperparameter Optimization (Hyperopt)
# ==========================================

def objective(params: dict) -> dict:
    """
    Objective function for Hyperopt to minimize. 
    Constructs a pipeline and calculates the RMSE of the ARDRegression model.

    Args:
        params (dict): Dictionary of hyperparameters provided by Hyperopt.

    Returns:
        dict: Dictionary containing the 'loss' (RMSE) and 'status'.
    """
    # 1. Extract hyperparameters
    # Note: 'k' must be an integer for SelectKBest
    k_features = int(params['k'])
    
    # 2. Define the Modeling Pipeline
    # Scaling is crucial for Bayesian linear models to ensure stable priors.
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=k_features)),
        ('regressor', ARDRegression(
            n_iter=1000,
            alpha_1=params['alpha_1'],
            alpha_2=params['alpha_2'],
            lambda_1=params['lambda_1'],
            lambda_2=params['lambda_2'],
            threshold_lambda=params['threshold_lambda']
        ))
    ])

    # 3. Data Preparation
    # 'trans' represents the training data slice for a specific category
    X = trans.drop(['Value', 'Category', 'Date'], axis=1, errors='ignore')
    y = trans['Value']

    # 4. Model Training and Evaluation
    # Note: In a production setting, Cross-Validation would be used here.
    pipeline.fit(X, y)
    y_pred = pipeline.predict(X)

    # Calculate Root Mean Squared Error (RMSE)
    rmse = mean_squared_error(y, y_pred, squared=False)

    return {'loss': rmse, 'status': STATUS_OK}

# 5. Define the Search Space
# We use log-uniform distributions or small uniform ranges for Bayesian priors
space = {
    'alpha_1': hp.uniform('alpha_1', 1e-9, 1e-1),
    'alpha_2': hp.uniform('alpha_2', 1e-9, 1e-1),
    'lambda_1': hp.uniform('lambda_1', 1e-9, 1e-1),
    'lambda_2': hp.uniform('lambda_2', 1e-9, 1e-1),
    'threshold_lambda': hp.uniform('threshold_lambda', 1e-5, 1),
    'k': hp.quniform('k', 2, 12, 1) # Selecting between 2 and 12 best lags
}

# --- Optimization Execution (Commented for Inference) ---
# trials = Trials()
# best_params = fmin(
#     fn=objective, 
#     space=space, 
#     algo=tpe.suggest, 
#     max_evals=1000, 
#     trials=trials
# )
# print("Best hyperparameters found:", best_params)

In [62]:
# Creating models with optimized hyperparameters obtained from optimization process
mod_trans = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=15)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.06888078283655773,alpha_2=0.03945852718958407,
                                     lambda_1=0.03702626329697483,threshold_lambda=0.9997701216269231,
                                     lambda_2=0.05734072355598921))])

mod_head = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=7)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.02481600128520766,alpha_2=0.008528655252303802,
                                     lambda_1=0.00014124088526198578,threshold_lambda=0.7450636107484564,
                                     lambda_2=0.09767294742641217))
        ])

mod_comm = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=15)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.07118099424945569,alpha_2=8.09972779238586e-05,
                                     lambda_1=2.1107382235219563e-06,threshold_lambda=0.5306309483437822,
                                     lambda_2=0.09976341436163934))])

mod_food= Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=14)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.0629442010193069,alpha_2=0.021658893986679657,
                                     lambda_1=0.01786690072203126,threshold_lambda=0.9554058127588239,
                                     lambda_2=0.09992888926123053))])

mod_clot =Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=11)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.09673709642511016,alpha_2=3.416909097873422e-05,
                                     lambda_1=9.980856718979543e-05,threshold_lambda=0.7288138072424017,
                                     lambda_2=0.0947889597342104))])

mod_alc = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=4)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.08966233855673965,alpha_2=0.006710715228320694,
                                     lambda_1=3.616958666849342e-06,threshold_lambda=0.8364030115930603,
                                     lambda_2=0.09999617314896067))])

mod_recre =Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=12)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.05255898747462261,alpha_2=3.7009918104093683e-05,
                                     lambda_1=5.271741774974271e-05,threshold_lambda=0.8326007995335035,
                                     lambda_2=0.09524135442298978))])

mod_health = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=12)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.027371582332002632,alpha_2=8.67637913797379e-05,
                                     lambda_1=3.174709090487235e-05,threshold_lambda=0.3157092294720313,
                                     lambda_2=0.0822705809211467))])

mod_hous = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=14)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.009737606644465512,alpha_2=0.00016445775438814932,
                                     lambda_1=1.7567631864029303e-05,threshold_lambda=0.8377464469150657,
                                     lambda_2=0.06138547431612553))])

mod_hhold = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=15)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.09994388977125979,alpha_2=0.0018078481292535741,
                                     lambda_1=4.718093183437776e-05,threshold_lambda=0.8250287362157941,
                                     lambda_2=0.09995632356972932))])

mod_rest = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=13)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.019251184865532744,alpha_2=0.015405006681598865,
                                     lambda_1=3.345488067675508e-05,threshold_lambda=0.9815688093826893,
                                     lambda_2= 0.09996117827416402))])

mod_edu = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=5)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.053009860661304836,alpha_2=2.5009337623756308e-5,
                                     lambda_1=3.539352154895098e-05,threshold_lambda=0.48364848313963765,
                                     lambda_2=0.09499504129208737))])

mod_Misc = Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(k=6)),
        ('classifier', ARDRegression(max_iter=1000,
                                     alpha_1=0.06764697343081885,alpha_2=0.0022580789917884233,
                                     lambda_1=0.006774595462953342,threshold_lambda=0.9988279228925528,
                                     lambda_2=0.0987091892989818))])

In [63]:
# ==========================================
# 9. Model Fitting Utility
# ==========================================

from sklearn.pipeline import Pipeline
from typing import Any

def fit_model_pipeline(model: Any, data: pd.DataFrame) -> Any:
    """
    Fits a provided model or pipeline to the given dataset.
    
    Automatically separates the target 'Value' from the features and 
    removes non-numeric metadata columns to prevent training errors.

    Args:
        model (sklearn.pipeline.Pipeline or BaseEstimator): An unfitted model or pipeline.
        data (pd.DataFrame): The training data containing features and the 'Value' target.

    Returns:
        Any: The fitted model/pipeline.
    """
    # Create a copy to prevent modifying the original dataframe
    train_df = data.copy()
    
    # Define features (X) and target (y)
    # We drop 'Value' (target) and metadata that shouldn't be used as features
    X = train_df.drop(columns=['Value', 'Category', 'Date'], errors='ignore')
    y = train_df['Value']
    
    # Fit the model
    model.fit(X, y)
    
    return model


In [64]:
# ==========================================
# 10. Per-Category Model Training
# ==========================================

# 1. Map each category to its specific pre-configured model
model_map = {
    'CPI Headline': mod_head,
    'Food and non alcoholic beverages': mod_food,
    'Alcoholic beverages and tobacco': mod_alc,
    'Clothing and footwear': mod_clot,
    'Housing and utilities': mod_hous,
    'Household contents and equipment': mod_hhold,
    'Health': mod_health,
    'Transport': mod_trans,
    'Communication': mod_comm,
    'Recreation and culture': mod_recre,
    'Education': mod_edu,
    'Restaurants and hotels': mod_rest,
    'Miscellaneous goods and services': mod_Misc
}

def train_all_categories(df: pd.DataFrame, models: dict) -> dict:
    """
    Iterates through each CPI category and fits a unique model for each.
    
    Args:
        df (pd.DataFrame): The dataframe containing lagged features for all categories.
        models (dict): A dictionary mapping category names to their model objects.
        
    Returns:
        dict: A dictionary of fitted model objects.
    """
    fitted_models = {}
    
    print("Starting training for all CPI categories...")
    
    for category_name, model_instance in models.items():
        # Slice the data for the specific category
        category_data = df[df['Category'] == category_name]
        
        if category_data.empty:
            print(f"Warning: No data found for category '{category_name}'. Skipping.")
            continue
            
        # Fit the model using our utility function
        # This function handles dropping 'Category' and 'Date' automatically
        fitted_model = fit_model_pipeline(model_instance, category_data)
        
        # Store the fitted model
        fitted_models[category_name] = fitted_model
        print(f"Successfully trained model for: {category_name}")
        
    return fitted_models

# Execute the training loop
trained_model_dict = train_all_categories(cpi_with_lags, model_map)

Starting training for all CPI categories...
Successfully trained model for: CPI Headline
Successfully trained model for: Food and non alcoholic beverages
Successfully trained model for: Alcoholic beverages and tobacco
Successfully trained model for: Clothing and footwear
Successfully trained model for: Housing and utilities
Successfully trained model for: Household contents and equipment
Successfully trained model for: Health
Successfully trained model for: Transport
Successfully trained model for: Communication
Successfully trained model for: Recreation and culture
Successfully trained model for: Education
Successfully trained model for: Restaurants and hotels
Successfully trained model for: Miscellaneous goods and services


# Preparing Submission

In [110]:
# ==========================================
# 11. Inference Preparation (August 2023)
# ==========================================

def prepare_inference_data(df: pd.DataFrame, target_month: str = '7-2023') -> pd.DataFrame:
    """
    Extracts the most recent feature row to be used for nowcasting the next month.
    
    Args:
        df (pd.DataFrame): The dataframe containing lagged features.
        target_month (str): The date of the last known observation (July 2023).

    Returns:
        pd.DataFrame: A dataframe containing the features for August 2023 forecast.
    """
    # 1. Filter for the most recent month (July 2023)
    # This row contains the 'Value' (July's CPI) and its lags (June, May, etc.)
    # We will shift these to become the features for August.
    latest_data = df[df['Date'] == target_month].copy()
    
    # 2. Re-aligning Lags for the future month:
    # To predict August:
    # August_Lag_1 = July_Value
    # August_Lag_2 = July_Lag_1 ... and so on.
    
    # Identify all 'Value_i' columns present in the training data
    lag_cols = [col for col in df.columns if col.startswith('Value_')]
    
    # Create the new feature set for August
    # The 'Value' from July becomes 'Value_1' for August.
    inference_row = latest_data[['Category', 'Value'] + lag_cols[:-1]].copy()
    
    # Rename columns to match the model's expected input (Value_1, Value_2, etc.)
    new_col_names = ['Category'] + [f'Value_{i}' for i in range(1, len(lag_cols) + 1)]
    inference_row.columns = new_col_names
    
    print(f"Inference features prepared for {len(inference_row)} categories.")
    return inference_row

# Note: Using the datetime object we created in step 6
inference_features = prepare_inference_data(cpi_with_lags, target_month='7-2023')

# Display features for a quick sanity check
inference_features.head()

Inference features prepared for 13 categories.


,Category,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,Value_8,Value_9,Value_10,Value_11,Value_12,Value_13,Value_14,Value_15
0,Miscellaneous goods and services,109.9,109.6,109.4,109.3,107.9,107.7,105.4,104.9,104.7,104.7,103.8,103.5,103.4,103.0,102.9
1,Restaurants and hotels,110.0,110.0,110.4,108.6,109.6,108.8,106.8,107.8,108.0,107.4,106.2,104.3,104.6,104.2,103.8
2,Education,110.4,110.4,110.4,110.4,110.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4,104.4
3,Recreation and culture,105.5,105.3,104.8,104.9,104.2,103.3,103.4,103.4,102.8,102.7,102.5,102.4,102.3,101.2,101.0
4,Communication,99.5,99.6,99.7,99.8,99.7,99.8,99.4,99.1,99.2,99.8,99.9,100.1,99.9,100.3,99.9


In [111]:
# ==========================================
# 12. Final Prediction & Submission Preparation
# ==========================================

def generate_nowcast(row: pd.Series, models: dict) -> float:
    """
    Retrieves the correct model for a category and generates a CPI prediction.

    Args:
        row (pd.Series): A single row of features including 'Category'.
        models (dict): The dictionary of fitted model objects.

    Returns:
        float: The predicted CPI value for August 2023.
    """
    category = row['Category']
    
    # Retrieve the model specific to this category
    model = models.get(category)
    
    if model is None:
        raise ValueError(f"No model found for category: {category}")

    # Prepare features: Drop the category label and reshape for scikit-learn (1, n_features)
    # .values.reshape(1, -1) is a clean way to handle single-row inference
    features = row.drop('Category').values.reshape(1, -1)
    
    # Generate prediction
    prediction = model.predict(features)
    
    # Return as a scalar float
    return float(prediction[0])

# 1. Execute the Nowcast
# We apply the prediction function to our prepared inference features
inference_features['Value'] = inference_features.apply(
    lambda x: generate_nowcast(x, trained_model_dict), axis=1
)

# 2. Map Categories to Competition Submission IDs
# This ensures the output matches the required Zindi submission format
SUBMISSION_ID_MAP = {
    'CPI Headline': 'August_headline CPI',
    'Food and non alcoholic beverages': 'August_food and non-alcoholic beverages',
    'Alcoholic beverages and tobacco': 'August_alcoholic beverages and tobacco',
    'Clothing and footwear': 'August_clothing and footwear',
    'Housing and utilities': 'August_housing and utilities',
    'Household contents and equipment': 'August_household contents and services',
    'Health': 'August_health',
    'Transport': 'August_transport',
    'Communication': 'August_communication',
    'Recreation and culture': 'August_recreation and culture',
    'Education': 'August_education',
    'Restaurants and hotels': 'August_restaurants and hotels',
    'Miscellaneous goods and services': 'August_miscellaneous goods and services'
}

# 3. Create the final submission dataframe
submission = pd.DataFrame({
    'ID': inference_features['Category'].map(SUBMISSION_ID_MAP),
    'Value': inference_features['Value']
})

# 4. Export for submission
submission.to_csv('South_Africa_CPI_Nowcast_August2023.csv', index=False)

print("Final predictions generated and saved to 'South_Africa_CPI_Nowcast_August2023.csv'.")
print(submission)

Final predictions generated and saved to 'South_Africa_CPI_Nowcast_August2023.csv'.
                                         ID       Value
0   August_miscellaneous goods and services  110.006993
1             August_restaurants and hotels  110.658906
2                          August_education  110.349224
3             August_recreation and culture  105.710773
4                      August_communication   99.533128
5                          August_transport  113.713328
6                             August_health  110.626064
7    August_household contents and services  108.166101
8              August_housing and utilities  108.430603
9              August_clothing and footwear  104.691877
10   August_alcoholic beverages and tobacco  111.762041
11  August_food and non-alcoholic beverages  119.403951
12                      August_headline CPI  111.353432
